# Headless Comfy — pipelines & LoRA stacks

Build/upload the 0.2.0 wheel from this repo first. This notebook uses new APIs that
are not present in 0.1.1. Restart the Colab runtime after changing installed versions.
GPU runtime required. Drive stores models/outputs; local disk caches model files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Install the published package release from GitHub, then upgrade the Colab CUDA stack.
%pip install https://github.com/merlinvn/headless-comfy/releases/download/v0.2.0/headless_comfy-0.2.0-py3-none-any.whl
%pip install -U torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu130

# Restart the Colab runtime after changing installed packages.


In [ ]:
import re
import subprocess
from pathlib import Path

def cache_file(src: Path, dst_dir: Path) -> Path:
    """Cache Drive file locally when missing or changed."""
    if not src.exists():
        raise FileNotFoundError(src)

    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / src.name

    needs_copy = (
        not dst.exists()
        or dst.stat().st_size != src.stat().st_size
        or dst.stat().st_mtime < src.stat().st_mtime
    )

    if needs_copy:
        print(f"\nCaching: {src.name}")
        subprocess.run(
            [
                "rsync",
                "-ah",
                "--info=progress2",
                str(src),
                str(dst),
            ],
            check=True,
        )
    else:
        print(f"Cached: {dst}")

    return dst

def assert_registered(kind: str, filename: str) -> None:
    available = hc.folder_paths.get_filename_list(kind)
    if filename not in available:
        raise RuntimeError(f"{filename!r} is not registered in {kind!r}")

def infer_lora_trigger(filename: str) -> str:
    """
    Examples:
      jun_vu_krea2_c1-st6000.safetensors -> jun_vu_x
      jun_vu_x_krea2_c1-st6000.safetensors -> jun_vu_x

    Fallback:
      abc_xyz.safetensors -> abc
    """
    stem = Path(filename).stem

    m = re.match(r"(.+?)_krea2_c\d+-st\d+$", stem)

    if m:
        name = m.group(1)
        # diem_my_9x is a good trigger, I don't want diem_my_9x_x
        return name if name.endswith("x") else f"{name}_x"

    # Match your old fallback behavior
    return filename.split("_")[0]

def cache_lora(name, strength=1.0, trigger=None, clip_strength=0.0):
    """Cache a Drive LoRA and construct its pipeline spec.

    None infers your filename trigger; an empty string disables the trigger.
    """
    cached = cache_file(DRIVE_ROOT / 'loras/krea2' / name, CACHE_ROOT / 'loras')
    assert_registered('loras', cached.name)
    return LoraSpec(
        name=cached.name, strength=strength,
        trigger=infer_lora_trigger(name) if trigger is None else trigger,
        clip_strength=clip_strength,
    )


def compact_lora_name(filename: str) -> str:
    """
    jun_vu_krea2_c1-st6000.safetensors
        -> jun_vu_st6000
    """

    stem = Path(filename).stem

    m = re.match(
        r"(.+?)_krea2_c\d+-st(\d+)$",
        stem,
    )

    if m:
        return (
            f"{m.group(1)}_"
            f"st{m.group(2)}"
        )

    return stem


def build_lora_stack(entry):
    """A filename/single spec creates one character; a named group combines characters."""
    if isinstance(entry, str):
        entry = {'name': entry}
    if not isinstance(entry, dict):
        raise TypeError('Each BATCH_LORAS entry must be a filename or dictionary')
    if 'loras' in entry:
        if not entry.get('name') or not entry['loras']:
            raise ValueError('A combination needs a name and at least one LoRA')
        members = entry['loras']
        stack_name = entry['name']
    else:
        members = [entry]
        stack_name = compact_lora_name(entry['name'])
    specs = []
    for member in members:
        if isinstance(member, str):
            member = {'name': member}
        specs.append(cache_lora(
            member['name'], strength=member.get('strength', 1.0),
            trigger=member.get('trigger'), clip_strength=member.get('clip_strength', 0.0),
        ))
    # Single files retain the old trigger folder; combinations get their own named folder.
    subdir = entry.get('output_subdir', stack_name if 'loras' in entry else specs[0].trigger)
    return LoraStack(stack_name, tuple(specs), output_subdir=subdir)


In [ ]:
import torch
from dataclasses import asdict
from headless_comfy import ComfyRuntime, LoraSpec, Pipeline, SamplingConfig
from headless_comfy.batch import BatchRunner, LoraStack, load_prompts
from headless_comfy.cli import doctor
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab')
doctor()
DRIVE_ROOT = Path('/content/drive/MyDrive/AI')
CACHE_ROOT = Path('/content/ai_cache')
MODEL = {
    'unet': {'src': DRIVE_ROOT / 'models/krea2/moodyKrea2MixUncensoredV8SEGoes_v80_int8.safetensors',
             'dst_dir': CACHE_ROOT / 'diffusion_models'},
    'clip': {'src': DRIVE_ROOT / 'models/qwen3vl_4b_fp8_scaled.safetensors',
             'dst_dir': CACHE_ROOT / 'text_encoders', 'type': 'krea2'},
    'vae': {'src': DRIVE_ROOT / 'models/qwen_image_vae.safetensors',
            'dst_dir': CACHE_ROOT / 'vae'},
}
LOCAL = {key: cache_file(cfg['src'], cfg['dst_dir']) for key, cfg in MODEL.items()}
hc = ComfyRuntime(CACHE_ROOT)
model = hc.load_diffusion_model(LOCAL['unet'].name)
clip = hc.load_clip(LOCAL['clip'].name, type=MODEL['clip']['type'])
vae = hc.load_vae(LOCAL['vae'].name)


## Configure LoRAs & pipeline
BATCH_LORAS below preserves the active files and commented choices from your old notebook.
Strength defaults to 1.0 when omitted; set strength on a member to override it.
Each BATCH_LORAS entry creates one variant: a filename for one character, a single
spec for a custom strength, or a named combination containing several weighted LoRAs.
Each variant is generated for every prompt and starts from the base model.
cache_lora() retains Drive caching and filename trigger inference.
Pass trigger='custom_word' to override or trigger='' for no trigger.

## Batch
Each entry in INPUT_OUTPUT runs all BATCH_LORAS variants against all prompts in that file.
Single-character outputs retain their trigger subfolder; combinations use their name.
RANDOM_ASPECT_RATIO=True chooses from ASPECT_RATIOS per image; False uses WIDTH/HEIGHT.
This applies to all_random and per_prompt seed modes. Chosen sizes persist for resume.
Images keep the old layout: OUTPUT_DIR / trigger / pXX_loraTag_seedNN.png.
Changing output_subdir on a stack chooses its output subfolder.
Prompts are archived to OUTPUT_DIR; the archived copy is used if the local file disappears.
Manifests live in a configuration-specific subfolder and persist random seeds/sizes.
SKIP_EXISTING=True uses the old filename-based logic: skip if any
pXX_loraTag_seed*.png exists in the output subfolder, including old notebook outputs.
It ignores changes to prompts, weights and sampling, and does not verify PNG integrity.
Set SKIP_EXISTING=False to generate again. This runner is single-process.


In [ ]:
BATCH_LORAS = [
    #  "hien_ho_krea2_c1-st4000.safetensors",
    # "hien_ho_krea2_c1-st5000.safetensors",
    # "hien_ho_krea2_c1-st6000.safetensors",

    # "ngoc_trinh_krea2_c1-st4000.safetensors",
    # "ngoc_trinh_krea2_c1-st5000.safetensors",
    # "ngoc_trinh_krea2_c1-st6000.safetensors",

    # "chu_an_krea2_c1-st4000.safetensors",
    # "chu_an_krea2_c1-st5000.safetensors",
    # "chu_an_krea2_c1-st6000.safetensors",

    # "diem_my_9x_krea2_c1-st4000.safetensors",
    # "diem_my_9x_krea2_c1-st5000.safetensors",
    # "diem_my_9x_krea2_c1-st6000.safetensors",

    # "jun_vu_krea2_c1-st4000.safetensors",
    # "jun_vu_krea2_c1-st5000.safetensors",
    # "jun_vu_krea2_c1-st6000.safetensors",

    # "midu_krea2_c1-st4000.safetensors",
    # "midu_krea2_c1-st5000.safetensors",
    # "midu_krea2_c1-st6000.safetensors",

    # "lth_krea2_c1-st4000.safetensors",
    # "lth_krea2_c1-st5000.safetensors",
    # "lth_krea2_c1-st6000.safetensors",

    # "lan_ngoc_krea2_c1-st4000.safetensors",
    # "lan_ngoc_krea2_c1-st5000.safetensors",
    # "lan_ngoc_krea2_c1-st6000.safetensors",

    # "sam_sam_krea2_c1-st4000.safetensors",
    # "sam_sam_krea2_c1-st5000.safetensors",
    # "sam_sam_krea2_c1-st6000.safetensors",

    # "chi_pu_krea2_c1-st4000.safetensors",
    # "chi_pu_krea2_c1-st5000.safetensors",
    # "chi_pu_krea2_c1-st6000.safetensors",

    # "lo_lem_krea2_c1-st4000.safetensors",
    # "lo_lem_krea2_c1-st5000.safetensors",
    # "lo_lem_krea2_c1-st6000.safetensors",

    # "Dilraba Dilmurat_000004000.safetensors",
    # "Liu Yifei_000004000.safetensors",
    "Zhao Liying_000004000.safetensors",
    "Lee Ju-eun_000004000.safetensors",

    # One character with its own strength:
    # {'name': 'char_a.safetensors', 'strength': 0.4},

    # One image variant combining three characters with independent strengths:
    # {
    #     'name': 'char_a_b_c',  # output folder and filename tag
    #     'loras': [
    #         {'name': 'char_a.safetensors', 'strength': 0.4},
    #         {'name': 'char_b.safetensors', 'strength': 0.2},
    #         {'name': 'char_c.safetensors', 'strength': 0.3},
    #     ],
    # },
]

# One entry = one variant, generated for every prompt in every INPUT_OUTPUT job.
# A combination applies all its LoRAs to the same image; each entry starts from the base.
STACKS = [build_lora_stack(entry) for entry in BATCH_LORAS]

BASE = SamplingConfig(steps=8)
HIRES = SamplingConfig(steps=4, denoise=0.39)
PIPELINE = Pipeline.two_pass(base=BASE, hires=HIRES, scale=1.5)

# Batch inputs, outputs and execution
INPUT_OUTPUT = {
    "/content/prompts_01.txt":
        DRIVE_ROOT / "outputs/krea2/20260917_batch_01",

    "/content/prompts_02.txt":
        DRIVE_ROOT / "outputs/krea2/20260917_batch_02",

    "/content/prompts_03.txt":
        DRIVE_ROOT / "outputs/krea2/20260919_batch_03",

    "/content/prompts_04.txt":
        DRIVE_ROOT / "outputs/krea2/20260919_batch_04",

    "/content/prompts_05.txt":
        DRIVE_ROOT / "outputs/krea2/20260920_batch_05",

    "/content/prompts_07.txt":
        DRIVE_ROOT / "outputs/krea2/20260920_batch_07",

    "/content/prompts_06.txt":
        DRIVE_ROOT / "outputs/krea2/20260920_batch_06",


}

SEED_MODE = 'all_random'  # or 'per_prompt'
SKIP_EXISTING = True  # Same filename-based skip logic as the old notebook
RANDOM_ASPECT_RATIO = True  #@param {type:"boolean"}
WIDTH = 800  #@param {type:"integer"}
HEIGHT = 1200  #@param {type:"integer"}
ASPECT_RATIOS = [
    (800, 1200),  # portrait
    (1200, 800),  # landscape
    (1024, 1024), # square
]
# Seed mode selects seeds; this flag selects random or fixed dimensions in either mode.
SIZES = tuple(ASPECT_RATIOS) if RANDOM_ASPECT_RATIO else ((WIDTH, HEIGHT),)
# Resume bookkeeping only: these IDs do NOT select LoRAs or output folders.
# They detect changed weights by source path, file size and modification time.
MODEL_IDS = {
    key: f'{cfg["src"]}:{cfg["src"].stat().st_size}:{cfg["src"].stat().st_mtime_ns}'
    for key, cfg in MODEL.items()
}
for stack in STACKS:
    for spec in stack.loras:
        source = DRIVE_ROOT / 'loras/krea2' / spec.name
        MODEL_IDS[f'lora:{spec.name}'] = f'{source}:{source.stat().st_size}:{source.stat().st_mtime_ns}'
runner = BatchRunner(hc, PIPELINE, model=model, clip=clip, vae=vae, model_ids=MODEL_IDS)
for prompt_file, output_dir in INPUT_OUTPUT.items():
    prompts = load_prompts(prompt_file, output_dir)
    results = runner.run(prompts, STACKS, output_dir, seed_mode=SEED_MODE,
                         sizes=SIZES, group_by_run=False, skip_existing=SKIP_EXISTING,
                         resume_mode="filename", on_result=lambda result: print(
                             'SKIP' if result['skipped'] else 'SAVED', result['path']))
    print('Generated:', sum(not r['skipped'] for r in results),
          'Skipped:', sum(r['skipped'] for r in results))


## Optional single-image preview
Run this final cell only when you want a preview. It uses the LoRAs and pipeline configured above.


In [ ]:
# Optional single preview uses the same pipeline as batch.
BASE_PROMPT = 'portrait photograph, soft natural light'
SEED = 32454
stack = STACKS[0]
model_lora, clip_lora = hc.apply_loras(model, stack.loras, clip)
state = PIPELINE.run(hc, model=model_lora, clip=clip_lora, vae=vae,
                     prompt=stack.prompt(BASE_PROMPT), seed=SEED, width=800, height=1200)
display(hc.tensor_to_pil(state.images))
del state, model_lora, clip_lora
hc.soft_empty_cache()
